# Pruebas Código 2: Precision/Recall, MMR, SPARQL y Query Transformation

Sistema RAG completo con evaluación de métricas, búsquedas diversas y transformación de consultas.

## Contenidos:
1. **Métricas**: Precision@k y Recall@k
2. **Búsquedas**: Similarity vs MMR
3. **SPARQL**: SELECT, FILTER, ORDER BY, LIMIT, UPDATE
4. **Query Transformation**: HyDE y Query Decomposition

---

In [2]:
# Setup inicial
import os, sys, time
from pathlib import Path
from typing import List, Dict, Set, Tuple
from datetime import datetime
import numpy as np
import pandas as pd
from collections import defaultdict

# Detectar raíz del repositorio
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
if not (REPO_ROOT / "src").exists():
    for parent in NOTEBOOK_DIR.parents:
        if (parent / "src").exists() and (parent / "requirements.txt").exists():
            REPO_ROOT = parent
            break

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import load_settings, init_embeddings, init_groq_llm
from src.graph import build_graph
from src.retrieval_metrics import recall_at_k, precision_at_k, evaluate_query_at_ks
from src.vectorstore import load_chroma_index
from src.query_transformer import transform_query

load_settings()
print('✅ Librerías importadas')
print(f'Hora: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


✅ Librerías importadas
Hora: 2026-03-28 21:15:53


## Sección 1: Precision y Recall Metrics

Implementar cálculo de Precision@k y Recall@k para evaluación de retrieval.

**Fórmulas:**
- Precision@k = TP / (TP + FP)
- Recall@k = TP / (TP + FN)


In [7]:
# Test 1: Precision y Recall Metrics
print('\n' + '='*80)
print('TEST 1: PRECISION Y RECALL METRICS')
print('='*80)

# Caso: Búsqueda de información sobre Ley 1010
test_query = "Información sobre la ley 1010 acoso laboral"
ground_truth_docs = ["LEY_1010_2006"]
eval_ks = (1, 3, 5)

# Simular resultado de búsqueda (ids de documentos recuperados)
retrieved_docs_at_k = {
    1: ["LEY_1010_2006"],  # Precision@1=1, Recall@1=1
    3: ["LEY_1010_2006", "DECRETO_1072_2015", "SENTENCIA_C310_2007"],  # 1 correcto de 3
    5: ["LEY_1010_2006", "DECRETO_1072_2015", "SENTENCIA_C310_2007", "LEY_50_1990", "DECRETO_36_2016"]  # 1 correcto de 5
}

print(f"\n📋 Consulta: {test_query}")
print(f"📋 Ground Truth: {ground_truth_docs}")
print(f"📋 Evaluando en k={eval_ks}\n")

for k in eval_ks:
    if k in retrieved_docs_at_k:
        retrieved = retrieved_docs_at_k[k]
        
        # Calcular métricas
        p_k = precision_at_k(retrieved, ground_truth_docs, k)
        r_k = recall_at_k(retrieved, ground_truth_docs, k)
        
        print(f"k={k}:")
        print(f"  Recuperados: {retrieved}")
        print(f"  ✓ Precision@{k}: {p_k:.3f} ({sum(1 for d in retrieved if d in ground_truth_docs)}/{k})")
        print(f"  ✓ Recall@{k}: {r_k:.3f} ({sum(1 for d in retrieved if d in ground_truth_docs)}/{len(ground_truth_docs)})")
        print()

# Usar la función evaluate_query_at_ks() con la firma actual (3 argumentos)
# Tomamos el ranking del k máximo para que la función calcule métricas en todos los ks por slicing.
max_k = max(eval_ks)
retrieved_for_eval = retrieved_docs_at_k[max_k]
metrics_result = evaluate_query_at_ks(retrieved_for_eval, ground_truth_docs, eval_ks)
print("📊 RESUMEN DE MÉTRICAS:")
for k, metrics in sorted(metrics_result.items()):
    print(f"  k={k}: Precision={metrics['precision']:.3f}, Recall={metrics['recall']:.3f}")

print('='*80)



TEST 1: PRECISION Y RECALL METRICS

📋 Consulta: Información sobre la ley 1010 acoso laboral
📋 Ground Truth: ['LEY_1010_2006']
📋 Evaluando en k=(1, 3, 5)

k=1:
  Recuperados: ['LEY_1010_2006']
  ✓ Precision@1: 1.000 (1/1)
  ✓ Recall@1: 1.000 (1/1)

k=3:
  Recuperados: ['LEY_1010_2006', 'DECRETO_1072_2015', 'SENTENCIA_C310_2007']
  ✓ Precision@3: 0.333 (1/3)
  ✓ Recall@3: 1.000 (1/1)

k=5:
  Recuperados: ['LEY_1010_2006', 'DECRETO_1072_2015', 'SENTENCIA_C310_2007', 'LEY_50_1990', 'DECRETO_36_2016']
  ✓ Precision@5: 0.200 (1/5)
  ✓ Recall@5: 1.000 (1/1)

[Métricas] Evaluando consulta individual con k=1...
[Métricas] Evaluando consulta individual con k=3...
[Métricas] Evaluando consulta individual con k=5...
📊 RESUMEN DE MÉTRICAS:
  k=1: Precision=1.000, Recall=1.000
  k=3: Precision=0.333, Recall=1.000
  k=5: Precision=0.200, Recall=1.000


## Sección 2: Maximal Marginal Relevance (MMR) Search

Implementar búsqueda MMR que equilibra relevancia y diversidad:

$$\text{MMR} = \arg\max_i [\lambda \cdot \text{Sim}(D_i, Q) - (1-\lambda) \cdot \max_j \text{Sim}(D_i, D_j)]$$

- $\lambda$: parámetro de balance (0=diversidad pura, 1=relevancia pura)
- Sim(D_i, Q): similitud documento-query
- Sim(D_i, D_j): similitud entre documentos seleccionados


In [8]:
# Test 2: Búsqueda con MMR vs Similarity
print('\n' + '='*80)
print('TEST 2: MMR SEARCH vs SIMILARITY SEARCH')
print('='*80)

try:
    # Cargar vectorstore
    persist_dir = os.getenv("CHROMA_PERSIST_DIR", "./data/chroma")
    collection_name = os.getenv("CHROMA_COLLECTION_NAME", "normativa_laboral")
    embedding_fn = init_embeddings()
    vectorstore = load_chroma_index(persist_dir, embedding_fn, collection_name)
    
    query = "acoso laboral Ley 1010"
    k = 3
    
    print(f"\n📋 Query: {query}")
    print(f"📋 k={k}\n")
    
    # Búsqueda: Similarity
    print("🔍 SIMILARITY SEARCH:")
    similarity_results = vectorstore.similarity_search_with_score(query, k=k)
    sim_ids = []
    for doc, score in similarity_results:
        doc_id = doc.metadata.get("id_documento", "N/A")
        sim_ids.append(doc_id)
        preview = doc.page_content[:80].replace('\n', ' ')
        print(f"  [{doc_id}] score={score:.3f} → {preview}...")
    
    # Búsqueda: MMR
    print("\n🔍 MMR SEARCH (λ=0.5, fetch_k=20):")
    mmr_results = vectorstore.max_marginal_relevance_search(
        query, 
        k=k,
        fetch_k=20,
        lambda_mult=0.5
    )
    mmr_ids = []
    for doc in mmr_results:
        doc_id = doc.metadata.get("id_documento", "N/A")
        mmr_ids.append(doc_id)
        preview = doc.page_content[:80].replace('\n', ' ')
        print(f"  [{doc_id}] → {preview}...")
    
    print("\n📊 COMPARACIÓN:")
    print(f"  Similarity IDs: {sim_ids}")
    print(f"  MMR IDs:        {mmr_ids}")
    
    diversidad = len(set(mmr_ids)) / len(mmr_ids) if mmr_ids else 0
    print(f"  Diversidad MMR: {diversidad:.1%} ({len(set(mmr_ids))} documentos únicos de {len(mmr_ids)})")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

print('='*80)



TEST 2: MMR SEARCH vs SIMILARITY SEARCH


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5267.70it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📋 Query: acoso laboral Ley 1010
📋 k=3

🔍 SIMILARITY SEARCH:
  [SENTENCIA_C139_2018] score=0.519 → Fecha ut supra DIANA FAJARDO RIVERA Magistrada ACLARACIÓN DE VOTO DEL MAGISTRADO...
  [SENTENCIA_C310_2007] score=0.523 → Salvamento DE VOTO DEL MAGISTRADO MANUEL JOSÉ CEPEDA ESPINOSA A LA SENTENCIA C-3...
  [SENTENCIA_C412_1997] score=0.532 → Ir al portal SUIN-Juriscol Ayúdanos a mejorar Imprimir la norma Responder Encues...

🔍 MMR SEARCH (λ=0.5, fetch_k=20):
  [SENTENCIA_C139_2018] → Fecha ut supra DIANA FAJARDO RIVERA Magistrada ACLARACIÓN DE VOTO DEL MAGISTRADO...
  [LEY_1010_2006] → Tratamiento sancionatorio al acoso laboral. El acoso laboral, cuando estuviere d...
  [SENTENCIA_C473_1994] → CASOS DE ILEGALIDAD Y SANCIONES. 1. La suspensión colectiva de trabajo es ilegal...

📊 COMPARACIÓN:
  Similarity IDs: ['SENTENCIA_C139_2018', 'SENTENCIA_C310_2007', 'SENTENCIA_C412_1997']
  MMR IDs:        ['SENTENCIA_C139_2018', 'LEY_1010_2006', 'SENTENCIA_C473_1994']
  Diversidad MMR: 100.0% (3 

## Sección 3: SPARQL Queries

SPARQL operaciones sobre RDF ontology de normativa laboral.

### 3.1 - SELECT Queries


In [22]:
# Test 3.1: SPARQL SELECT
print('\n' + '='*80)
print('TEST 3.1: SPARQL SELECT QUERY')
print('='*80)

try:
    from src.ontology.graphdb_client import GraphDBClient
    
    graphdb = GraphDBClient()
    
    # SELECT: consultar recursos clave de la ontología
    select_query = """
    PREFIX ex: <http://example.org/ontologia-laboral#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT ?recurso ?tipo ?nombre ?label
    WHERE {
        VALUES ?recurso { ex:OntologiaLaboral ex:actor_001 ex:actor_002 ex:actor_003 }
        ?recurso a ?tipo .
        OPTIONAL { ?recurso ex:tieneNombre ?nombre . }
        OPTIONAL { ?recurso rdfs:label ?label . }
    }
    ORDER BY ?recurso ?tipo
    LIMIT 20
    """
    
    print("\n📋 SPARQL SELECT Query:")
    print("  SELECT sobre ex:OntologiaLaboral y ex:actor_001..003")
    
    result = graphdb.select(select_query)
    
    if result:
        print(f"\n✅ Resultados: {len(result)} filas")
        df = pd.DataFrame(result)
        print(df.to_string())
    else:
        print("\n⚠️ Sin resultados (ontología podría estar vacía)")
    
except Exception as e:
    print(f"ℹ️ SPARQL no disponible o ontología vacía: {e}")
    print("   (Esto es esperado si no hay GraphDB configurado)")

print('='*80)



TEST 3.1: SPARQL SELECT QUERY

📋 SPARQL SELECT Query:
  SELECT sobre ex:OntologiaLaboral y ex:actor_001..003

✅ Resultados: 9 filas
                                                 recurso                                               tipo               nombre                                         label
0  http://example.org/ontologia-laboral#OntologiaLaboral      http://www.w3.org/2000/01/rdf-schema#Resource                 None  Ontologia OWL del Dominio Laboral Colombiano
1  http://example.org/ontologia-laboral#OntologiaLaboral             http://www.w3.org/2002/07/owl#Ontology                 None  Ontologia OWL del Dominio Laboral Colombiano
2  http://example.org/ontologia-laboral#OntologiaLaboral                http://www.w3.org/2002/07/owl#Thing                 None  Ontologia OWL del Dominio Laboral Colombiano
3         http://example.org/ontologia-laboral#actor_001  http://example.org/ontologia-laboral#ActorLaboral     Actor Sindical A                                       

### 3.2 - FILTER Operations


In [23]:
# Test 3.2: SPARQL FILTER
print('\n' + '='*80)
print('TEST 3.2: SPARQL FILTER')
print('='*80)

try:
    # FILTER: actores laborales cuyo nombre contiene 'actor'
    filter_query = """
    PREFIX ex: <http://example.org/ontologia-laboral#>
    
    SELECT ?actor ?nombre
    WHERE {
        ?actor a ex:ActorLaboral ;
               ex:tieneNombre ?nombre .
        FILTER(CONTAINS(LCASE(STR(?nombre)), "actor"))
    }
    ORDER BY ?nombre
    """
    
    print("\n📋 SPARQL FILTER Query (nombre contiene 'actor'):")
    print("  SELECT ?actor ?nombre WHERE { ?actor a ex:ActorLaboral ... FILTER(...) }")
    
    result = graphdb.select(filter_query)
    
    if result:
        print(f"\n✅ Resultados: {len(result)} actores encontrados")
        df = pd.DataFrame(result)
        print(df.to_string())
    else:
        print("\n⚠️ Sin resultados")
    
except Exception as e:
    print(f"ℹ️ SPARQL FILTER no disponible: {e}")

print('='*80)



TEST 3.2: SPARQL FILTER

📋 SPARQL FILTER Query (nombre contiene 'actor'):
  SELECT ?actor ?nombre WHERE { ?actor a ex:ActorLaboral ... FILTER(...) }



✅ Resultados: 4 actores encontrados
                                            actor               nombre
0  http://example.org/ontologia-laboral#actor_002  Actor Empresarial B
1  http://example.org/ontologia-laboral#actor_003      Actor Publico C
2  http://example.org/ontologia-laboral#actor_001     Actor Sindical A
3  http://example.org/ontologia-laboral#actor_004       Actor Social D


### 3.3 - ORDER BY y LIMIT


In [24]:
# Test 3.3: SPARQL ORDER BY y LIMIT
print('\n' + '='*80)
print('TEST 3.3: SPARQL ORDER BY y LIMIT')
print('='*80)

try:
    # ORDER BY DESC(?anio) LIMIT 5 sobre normas jurídicas
    order_limit_query = """
    PREFIX ex: <http://example.org/ontologia-laboral#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    
    SELECT ?norma ?nombre ?anio
    WHERE {
        ?norma a ex:NormaJuridica ;
               ex:tieneNombre ?nombre ;
               ex:tieneAnioPublicacion ?anio .
    }
    ORDER BY DESC(?anio)
    LIMIT 5
    """
    
    print("\n📋 SPARQL ORDER BY DESC(?anio) LIMIT 5:")
    print("  SELECT ?norma ?nombre ?anio WHERE { ?norma a ex:NormaJuridica ... }")
    
    result = graphdb.select(order_limit_query)
    
    if result:
        print(f"\n✅ Top 5 normas más recientes:")
        df = pd.DataFrame(result)
        print(df.to_string())
    else:
        print("\n⚠️ Sin resultados")
    
except Exception as e:
    print(f"ℹ️ SPARQL ORDER BY/LIMIT no disponible: {e}")

print('='*80)



TEST 3.3: SPARQL ORDER BY y LIMIT

📋 SPARQL ORDER BY DESC(?anio) LIMIT 5:
  SELECT ?norma ?nombre ?anio WHERE { ?norma a ex:NormaJuridica ... }

✅ Top 5 normas más recientes:
                                                    norma                             nombre  anio
0      http://example.org/ontologia-laboral#ley_2209_2022                   Ley 2209 de 2022  2022
1      http://example.org/ontologia-laboral#ley_2088_2021                   Ley 2088 de 2021  2021
2      http://example.org/ontologia-laboral#ley_2114_2021                   Ley 2114 de 2021  2021
3  http://example.org/ontologia-laboral#norma_general_004                   Convenio OIT 187  2006
4  http://example.org/ontologia-laboral#norma_general_002  Constitucion Politica de Colombia  1991


### 3.4 - UPDATE Operations (INSERT y DELETE)


In [25]:
# Test 3.4: SPARQL UPDATE (INSERT + DELETE)
print('\n' + '='*80)
print('TEST 3.4: SPARQL UPDATE (INSERT y DELETE)')
print('='*80)

try:
    # UPDATE 1: INSERT (agregar actor laboral de prueba)
    insert_query = """
    PREFIX ex: <http://example.org/ontologia-laboral#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    
    INSERT DATA {
        ex:actor_prueba_2026 a ex:ActorLaboral, owl:Thing ;
                           ex:tieneNombre "Actor de Prueba 2026" .
    }
    """
    
    print("\n📝 UPDATE 1: SPARQL INSERT DATA")
    print("  INSERT DATA { ex:actor_prueba_2026 a ex:ActorLaboral ; ex:tieneNombre ... }")
    
    try:
        graphdb.update(insert_query)
        print("  ✅ Insertado: actor_prueba_2026")
    except Exception as e:
        print(f"  ℹ️ INSERT no ejecutado: {str(e)[:100]}")
    
    # UPDATE 2: DELETE + INSERT (modificar nombre del actor)
    delete_insert_query = """
    PREFIX ex: <http://example.org/ontologia-laboral#>
    
    DELETE {
        ex:actor_prueba_2026 ex:tieneNombre "Actor de Prueba 2026" .
    }
    INSERT {
        ex:actor_prueba_2026 ex:tieneNombre "Actor de Prueba 2026 (actualizado)" .
    }
    WHERE {
        ex:actor_prueba_2026 ex:tieneNombre "Actor de Prueba 2026" .
    }
    """
    
    print("\n📝 UPDATE 2: SPARQL DELETE + INSERT")
    print("  DELETE { ex:actor_prueba_2026 ex:tieneNombre \"Actor de Prueba 2026\" }")
    print("  INSERT { ex:actor_prueba_2026 ex:tieneNombre \"Actor de Prueba 2026 (actualizado)\" }")
    
    try:
        graphdb.update(delete_insert_query)
        print("  ✅ Modificado: nombre del actor actualizado")
    except Exception as e:
        print(f"  ℹ️ DELETE+INSERT no ejecutado: {str(e)[:100]}")
    
    # Verificar cambios
    verify_query = """
    PREFIX ex: <http://example.org/ontologia-laboral#>
    
    SELECT ?nombre
    WHERE {
        ex:actor_prueba_2026 ex:tieneNombre ?nombre .
    }
    """
    
    print("\n✓ Verificación:")
    result = graphdb.select(verify_query)
    if result:
        for row in result:
            print(f"  actor_prueba_2026: nombre={row.get('nombre')}")
    
except Exception as e:
    print(f"ℹ️ SPARQL UPDATE no disponible: {e}")

print('='*80)



TEST 3.4: SPARQL UPDATE (INSERT y DELETE)

📝 UPDATE 1: SPARQL INSERT DATA
  INSERT DATA { ex:actor_prueba_2026 a ex:ActorLaboral ; ex:tieneNombre ... }
  ✅ Insertado: actor_prueba_2026

📝 UPDATE 2: SPARQL DELETE + INSERT
  DELETE { ex:actor_prueba_2026 ex:tieneNombre "Actor de Prueba 2026" }
  INSERT { ex:actor_prueba_2026 ex:tieneNombre "Actor de Prueba 2026 (actualizado)" }
  ✅ Modificado: nombre del actor actualizado

✓ Verificación:
  actor_prueba_2026: nombre=Actor de Prueba 2026 (actualizado)


## Sección 4: Query Transformation

### 4.1 - HyDE (Hypothetical Document Embeddings)

Para preguntas cortas, expandir contexto mediante documento hipotético.


In [19]:
# Test 4.1: HyDE (Pregunta corta)
print('\n' + '='*80)
print('TEST 4.1: HyDE - QUERY TRANSFORMATION (Pregunta Corta)')
print('='*80)

try:
    # Pregunta corta
    short_query = "¿Qué es acoso laboral?"
    
    print(f"\n📋 Pregunta original (corta): {short_query}")
    
    # Cargar componentes
    persist_dir = os.getenv("CHROMA_PERSIST_DIR", "./data/chroma")
    collection_name = os.getenv("CHROMA_COLLECTION_NAME", "normativa_laboral")
    embedding_fn = init_embeddings()
    vectorstore = load_chroma_index(persist_dir, embedding_fn, collection_name)
    llm = init_groq_llm(temperature=0.2)
    
    # Transformación + retrieval en una sola llamada
    result = transform_query(short_query, llm=llm, vectorstore=vectorstore, k=3)
    
    print(f"\n✨ Transformación aplicada:")
    print(f"  Tipo: {result.get('query_type', 'N/A')}")
    
    transformed_queries = result.get('transformed_queries', [])
    hypothesis = transformed_queries[0] if transformed_queries else ''
    print(f"  Documento hipotético generado ({len(hypothesis)} chars):")
    
    # Mostrar preview del documento hipotético
    lines = hypothesis.split('\n')[:5]
    for line in lines:
        if line.strip():
            print(f"    {line[:100]}...")
    
    print(f"\n🔍 Resultados con HyDE:")
    print("  Resultados encontrados:")
    for doc in result.get('documents', []):
        doc_id = doc.metadata.get("id_documento", "N/A")
        preview = doc.page_content[:80].replace('\n', ' ')
        print(f"    [{doc_id}] → {preview}...")
    
except Exception as e:
    print(f"❌ Error en HyDE: {e}")

print('='*80)



TEST 4.1: HyDE - QUERY TRANSFORMATION (Pregunta Corta)

📋 Pregunta original (corta): ¿Qué es acoso laboral?


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4985.97it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✨ Transformación de consulta: HYDE
   Consulta original: ¿Qué es acoso laboral?

📚 HyDE: Generando documento hipotético...
   ✓ Documento hipotético generado (1170 caracteres)
   Preview: El acoso laboral se refiere a un patrón de comportamiento hostil y repetitivo que un empleado experi...
   ✓ Recuperados 3 documentos

✨ Transformación aplicada:
  Tipo: hyde
  Documento hipotético generado (1170 chars):
    El acoso laboral se refiere a un patrón de comportamiento hostil y repetitivo que un empleado experi...

🔍 Resultados con HyDE:
  Resultados encontrados:
    [LEY_1010_2006] → intimidación, terror y angustia, a causar perjuicio laboral, generar desmotivaci...
    [LEY_1010_2006] → a) Los actos de agresión física, independientemente de sus consecuencias;     b)...
    [LEY_1010_2006] → Tratamiento sancionatorio al acoso laboral. El acoso laboral, cuando estuviere d...


### 4.2 - Query Decomposition (Preguntas Múltiples)


In [20]:
# Test 4.2: Query Decomposition (Pregunta múltiple)
print('\n' + '='*80)
print('TEST 4.2: QUERY DECOMPOSITION - (Pregunta Múltiple)')
print('='*80)

try:
    # Pregunta múltiple/compleja
    complex_query = "¿Cuáles son las diferencias entre la Ley 1010 y el Decreto 1072 sobre acoso laboral? ¿Qué sanciones establece cada una?"
    
    print(f"\n📋 Pregunta original (compleja):")
    print(f"  {complex_query}\n")
    
    # Cargar componentes
    persist_dir = os.getenv("CHROMA_PERSIST_DIR", "./data/chroma")
    collection_name = os.getenv("CHROMA_COLLECTION_NAME", "normativa_laboral")
    embedding_fn = init_embeddings()
    vectorstore = load_chroma_index(persist_dir, embedding_fn, collection_name)
    llm = init_groq_llm(temperature=0.2)
    
    # Transformación + retrieval
    result = transform_query(complex_query, llm=llm, vectorstore=vectorstore, k=4)
    
    print(f"✨ Transformación aplicada:")
    print(f"  Tipo: {result.get('query_type', 'N/A')}")
    
    if result.get('query_type') == 'decomposition':
        sub_queries = result.get('transformed_queries', [])
        print(f"\n📌 Sub-queries identificadas ({len(sub_queries)}):")
        for i, subq in enumerate(sub_queries, 1):
            print(f"  [{i}] {subq}")
    else:
        print(f"  (Sistema detectó como: {result.get('query_type')})")
    
    print(f"\n🔍 Resultados recuperados:")
    for doc in result.get('documents', []):
        doc_id = doc.metadata.get("id_documento", "N/A")
        preview = doc.page_content[:90].replace('\n', ' ')
        print(f"  ✓ [{doc_id}] → {preview}...")
    
except Exception as e:
    print(f"❌ Error en Query Decomposition: {e}")

print('='*80)



TEST 4.2: QUERY DECOMPOSITION - (Pregunta Múltiple)

📋 Pregunta original (compleja):
  ¿Cuáles son las diferencias entre la Ley 1010 y el Decreto 1072 sobre acoso laboral? ¿Qué sanciones establece cada una?



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4605.36it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✨ Transformación de consulta: DECOMPOSITION
   Consulta original: ¿Cuáles son las diferencias entre la Ley 1010 y el Decreto 1072 sobre acoso laboral? ¿Qué sanciones establece cada una?

🔗 Query Decomposition: Analizando consulta compleja...
   ✓ Consulta descompuesta en 5 sub-consultas:
      1. ¿Qué es la Ley 1010 sobre acoso laboral?
      2. ¿Qué es el Decreto 1072 sobre acoso laboral?
      3. ¿Cuáles son las diferencias clave entre la Ley 1010 y el Decreto 1072 sobre acoso laboral?
      4. ¿Qué sanciones establece la Ley 1010 para el acoso laboral?
      5. ¿Qué sanciones establece el Decreto 1072 para el acoso laboral?

🔎 Ejecutando búsquedas para cada sub-consulta...
   [1/5] Buscando: ¿Qué es la Ley 1010 sobre acoso laboral?
      ✓ 4 documentos recuperados
   [2/5] Buscando: ¿Qué es el Decreto 1072 sobre acoso laboral?
      ✓ 4 documentos recuperados
   [3/5] Buscando: ¿Cuáles son las diferencias clave entre la Ley 1010 y el Decreto 1072 sobre acoso laboral?
      ✓ 4 docu

## Sección 5: Pipeline Completo con Tools del Sistema

En esta sección se prueban 5 escenarios end-to-end del pipeline LangGraph:

1. **5.1** Búsqueda por tipo de documento (`search_by_document_type`)
2. **5.2** Búsqueda por rango de años (`search_by_year_range`)
3. **5.3** Extracción de artículo específico (`extract_specific_article`)
4. **5.4** Comparación entre documentos (`compare_documents`)
5. **5.5** Resumen de documento (`resume_document`)

Cada test ejecuta: **classify -> query_transform -> tool_calling -> retrieve -> kg_retrieve -> generate -> verify**.

In [29]:
# Test 5.1: Pipeline completo - search_by_document_type
print('\n' + '='*80)
print('TEST 5.1: PIPELINE COMPLETO - SEARCH_BY_DOCUMENT_TYPE')
print('='*80)

try:
    # Reutilizar el grafo si ya existe
    if 'pipeline_graph' not in globals():
        load_settings()
        pipeline_graph = build_graph()
    
    def run_pipeline_test(query: str, title: str):
        print(f"\n🧪 {title}")
        print(f"📋 Query: {query}")
        
        initial_state = {
            'query': query,
            'classification': '',
            'query_type': None,
            'transformed_queries': None,
            'documents': [],
            'tool_results': None,
            'kg_results': None,
            'answer': '',
            'verification': {},
            'metadata': {}
        }
        
        try:
            result = pipeline_graph.invoke(initial_state)
            tool_used = (result.get('tool_results') or {}).get('tool_used', 'none') if isinstance(result.get('tool_results'), dict) else 'none'
            docs = result.get('documents', [])
            
            print(f"  ✓ Clasificación: {result.get('classification', 'N/A')}")
            print(f"  ✓ Query type: {result.get('query_type', 'N/A')}")
            print(f"  ✓ Tool usada: {tool_used}")
            print(f"  ✓ Docs recuperados: {len(docs)}")
            
            answer = result.get('answer', '')
            print(f"  ✓ Respuesta (preview): {answer[:220]}...")
            return result
        except BaseException as pipeline_error:
            print(f"  ❌ Error en ejecución del pipeline: {pipeline_error}")
            return {'error': str(pipeline_error), 'query': query, 'title': title}
    
    q_51 = 'Muéstrame información de la Ley 50 de 1990 sobre obligaciones laborales'
    result_51 = run_pipeline_test(q_51, '5.1 Búsqueda por tipo de documento')
    
except Exception as e:
    print(f"❌ Error en test 5.1: {e}")

print('='*80)


TEST 5.1: PIPELINE COMPLETO - SEARCH_BY_DOCUMENT_TYPE

🧪 5.1 Búsqueda por tipo de documento
📋 Query: Muéstrame información de la Ley 50 de 1990 sobre obligaciones laborales

[CLASSIFY] Muéstrame información de la Ley 50 de 1990 sobre obligaciones laborales
   [WARN] Error en clasificacion (fallback simple): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in o
   [OK] Clasificacion (fallback): legal_specific

[TRANSFORM] Iniciada


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4006.35it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✨ Transformación de consulta: HYDE
   Consulta original: Muéstrame información de la Ley 50 de 1990 sobre obligaciones laborales

📚 HyDE: Generando documento hipotético...
   ✓ Documento hipotético generado (966 caracteres)
   Preview: La Ley 50 de 1990 establece disposiciones laborales importantes en Colombia, enfocándose en la regul...
   ✓ Recuperados 4 documentos
   [OK] Tipo: hyde
   [OK] Consultas: 1

🔧 EVALUANDO HERRAMIENTAS ESPECIALIZADAS (ReAct + bind_tools)
   [WARN] Error en ReAct: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01km8dywpgec5a2fazs9kz2t3f` servi
   [WARN] Usando fallback
   [OK] Tool: search_by_document_type

📚 RECUPERANDO DOCUMENTOS


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5649.37it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ⚠️ select_dynamic_k falló: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01km8dywpgec5a2fazs9kz2t3f` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99993, Requested 218. Please try again in 3m2.304s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}} — aplicando fallback
   🎯 Búsqueda con metadata: LEY 50
      ID objetivo: LEY_50_1990
   [OK] 5 docs recuperados
      1. LEY_50_1990 (LEY) - score: 0.4646
      2. LEY_50_1990 (LEY) - score: 0.5023
      3. LEY_50_1990 (LEY) - score: 0.5293
      4. LEY_50_1990 (LEY) - score: 0.5319
      5. LEY_50_1990 (LEY) - score: 0.5399
   [MTR] Retrieval metrics omitidas: falta ground truth

[KG] Recuperando contexto estructurado (GraphDB)
   [OK] Filas KG: 5

[GEN] Generando respuesta
   [WARN] Error en generacion: Error code: 429 - {'error': {'message': 'Rate limi

In [30]:
# Test 5.2: Pipeline completo - search_by_year_range
print('\n' + '='*80)
print('TEST 5.2: PIPELINE COMPLETO - SEARCH_BY_YEAR_RANGE')
print('='*80)

try:
    q_52 = '¿Qué normas laborales fueron publicadas entre 1990 y 2022 sobre derechos y obligaciones?'
    result_52 = run_pipeline_test(q_52, '5.2 Búsqueda por rango de años')
except Exception as e:
    print(f"❌ Error en test 5.2: {e}")

print('='*80)


TEST 5.2: PIPELINE COMPLETO - SEARCH_BY_YEAR_RANGE

🧪 5.2 Búsqueda por rango de años
📋 Query: ¿Qué normas laborales fueron publicadas entre 1990 y 2022 sobre derechos y obligaciones?

[CLASSIFY] ¿Qué normas laborales fueron publicadas entre 1990 y 2022 sobre derechos y obligaciones?
   [WARN] Error en clasificacion (fallback simple): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in o
   [OK] Clasificacion (fallback): general_laboral

[TRANSFORM] Iniciada


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4198.02it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✨ Transformación de consulta: HYDE
   Consulta original: ¿Qué normas laborales fueron publicadas entre 1990 y 2022 sobre derechos y obligaciones?

📚 HyDE: Generando documento hipotético...
   ⚠️ Error en HyDE: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01km8dywpgec5a2fazs9kz2t3f` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99976, Requested 118. Please try again in 1m21.216s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
   [OK] Tipo: hyde
   [OK] Consultas: 1

🔧 EVALUANDO HERRAMIENTAS ESPECIALIZADAS (ReAct + bind_tools)
   [WARN] Error en ReAct: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01km8dywpgec5a2fazs9kz2t3f` servi
   [WARN] Usando fallback
   [OK] Tool: search_by_year_range

📚 RECUPERANDO DOCUMENTOS


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9516.86it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ⚠️ select_dynamic_k falló: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01km8dywpgec5a2fazs9kz2t3f` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99968, Requested 223. Please try again in 2m45.024s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}} — aplicando fallback
   🔧 Ejecutando: search_by_year_range
      [OK] 5 docs en rango 1990-2022
   [OK] 5 docs recuperados
      1. SENTENCIA_C988_1999 (SENTENCIA) - [herramienta]
      2. SENTENCIA_C72_1994 (SENTENCIA) - [herramienta]
      3. SENTENCIA_C42_2003 (SENTENCIA) - [herramienta]
      4. SENTENCIA_C471_2020 (SENTENCIA) - [herramienta]
      5. SENTENCIA_C72_1994 (SENTENCIA) - [herramienta]
   [MTR] Retrieval metrics omitidas: falta ground truth

[KG] Recuperando contexto estructurado (GraphDB)
   [OK] Filas KG: 5

[GEN] Generando respuesta
   [

In [ ]:
# Test 5.3: Pipeline completo - extract_specific_article
print('\n' + '='*80)
print('TEST 5.3: PIPELINE COMPLETO - EXTRACT_SPECIFIC_ARTICLE')
print('='*80)

try:
    q_53 = '¿Qué dice el artículo 5 de la Ley 1010 de 2006?'
    result_53 = run_pipeline_test(q_53, '5.3 Extracción de artículo específico')
except Exception as e:
    print(f"❌ Error en test 5.3: {e}")

print('='*80)


TEST 5.3: PIPELINE COMPLETO - EXTRACT_SPECIFIC_ARTICLE

🧪 5.3 Extracción de artículo específico
📋 Query: ¿Qué dice el artículo 5 de la Ley 1010 de 2006?

[CLASSIFY] ¿Qué dice el artículo 5 de la Ley 1010 de 2006?
   [WARN] Error en clasificacion (fallback simple): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in o
   [OK] Clasificacion (fallback): legal_specific

[TRANSFORM] Iniciada


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4670.90it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✨ Transformación de consulta: HYDE
   Consulta original: ¿Qué dice el artículo 5 de la Ley 1010 de 2006?

📚 HyDE: Generando documento hipotético...


In [1]:
# Test 5.4: Pipeline completo - compare_documents
print('\n' + '='*80)
print('TEST 5.4: PIPELINE COMPLETO - COMPARE_DOCUMENTS')
print('='*80)

try:
    q_54 = '¿Cuáles son las diferencias entre la Ley 1010 y el Decreto 1072 sobre acoso laboral?'
    result_54 = run_pipeline_test(q_54, '5.4 Comparación de documentos')
except Exception as e:
    print(f"❌ Error en test 5.4: {e}")

print('='*80)


TEST 5.4: PIPELINE COMPLETO - COMPARE_DOCUMENTS
❌ Error en test 5.4: name 'run_pipeline_test' is not defined


In [2]:
# Test 5.5: Pipeline completo - resume_document
print('\n' + '='*80)
print('TEST 5.5: PIPELINE COMPLETO - RESUME_DOCUMENT')
print('='*80)

try:
    q_55 = 'Dame un resumen del Decreto 36 de 2016'
    result_55 = run_pipeline_test(q_55, '5.5 Resumen de documento')
except Exception as e:
    print(f"❌ Error en test 5.5: {e}")

print('='*80)


TEST 5.5: PIPELINE COMPLETO - RESUME_DOCUMENT
❌ Error en test 5.5: name 'run_pipeline_test' is not defined
